# El panel y la etiqueta: qué es un «inicio» y cuántos hay

El informe no predice si habrá muchos o pocos casos: predice el **inicio** de
un brote, que es distinto de su continuación. Este cuaderno describe el panel
publicado y caracteriza esa etiqueta, que es la que decide todo lo demás.

**Qué necesita.** Solo `datos/panel/` y el entorno del proyecto
(`uv sync --locked --project pipeline`). No usa predicciones ni modelos.

**Cuánto tarda.** Menos de un minuto.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "pipeline" / "src"))

from sat_dengue_pipeline.evaluation import validacion_anual as va  # noqa: E402

PANEL = RAIZ / "datos" / "panel"

esquema = json.loads((PANEL / "esquema.json").read_text(encoding="utf-8"))
panel = pd.read_csv(PANEL / "panel_meta_tolima.csv", dtype=object)
for columna in esquema["columns"]:
    panel[columna["name"]] = panel[columna["name"]].astype(columna["dtype"])
print(f"{len(panel)} filas | {panel.municipality_code.nunique()} municipios "
      f"| {panel.year_month.min()} a {panel.year_month.max()}")

## 1. Qué hay en el panel

Cada fila es un municipio en un mes. Las columnas se agrupan por rol: claves,
metadatos, predictores (los que el modelo puede ver al cierre del mes de
origen), objetivos y controles.

In [ ]:
roles = esquema["column_roles"]
pd.DataFrame(
    [{"rol": rol, "columnas": len(nombres), "ejemplos": ", ".join(nombres[:4])}
     for rol, nombres in roles.items()]
)

El canal endémico necesita cinco años previos del mismo mes calendario, así
que las columnas que dependen de él están vacías al principio de la serie.
Esto no es un defecto de los datos: es la razón por la que la evaluación
empieza en 2012 y no en 2007.

In [ ]:
cobertura = (
    panel.assign(anio=panel.year_month.str[:4])
    .groupby("anio")
    .agg(filas=("municipality_code", "size"),
         con_canal=("p75_canal_t", lambda s: int(s.notna().sum())),
         casos=("dengue_notifications_count", "sum"))
)
cobertura.head(8)

## 2. La etiqueta: inicio contra continuación

Un mes está **en exceso** cuando sus casos superan el percentil 75 del mismo
mes calendario en los cinco años anteriores del propio municipio. Un **inicio**
es el primer mes en exceso después de uno que no lo estaba. Todo lo demás que
está en exceso es continuación.

La distinción importa porque un episodio de exceso dura varios meses: un modelo
puede sacar buenas métricas agregadas limitándose a reconocer brotes que ya
están en curso, que son justamente los que la autoridad sanitaria ya conoce.

In [ ]:
historia = {h: va.historia_desde_panel(panel, h) for h in (1, 2)}
resumen = pd.DataFrame(
    [
        {
            "horizonte": f"T+{h}",
            "filas con etiqueta": len(d),
            "meses en exceso (objetivo)": int(((d.inicio == 1) | (d.exceso_t == 1)).sum()),
            "inicios": int((d.inicio == 1).sum()),
            "cohorte evaluable (fuera de exceso en el origen)": int((d.exceso_t == 0).sum()),
            "inicios en esa cohorte": int(((d.inicio == 1) & (d.exceso_t == 0)).sum()),
        }
        for h, d in historia.items()
    ]
)
resumen

La **cohorte evaluable** son los municipios que no están en exceso en el mes de
origen, que es donde un inicio es posible. Es sobre esas filas, y no sobre todo
el panel, que se mide la sensibilidad del informe.

In [ ]:
inicios_anio = pd.concat(
    [
        d[(d.exceso_t == 0) & d.anio_objetivo.between(2018, 2024)]
        .groupby("anio_objetivo")
        .inicio.sum()
        .rename(f"T+{h}")
        for h, d in historia.items()
    ],
    axis=1,
)
inicios_anio.loc["total"] = inicios_anio.sum()
inicios_anio

## 3. De dónde arrancan los inicios

Esta es la pregunta que más dice sobre el techo del problema: cuántos inicios
empiezan en un municipio que **no notificó ningún caso** en el mes de origen.
Ahí un sistema basado en conteos no tiene señal que ordenar, porque la entrada
es cero para todos los candidatos.

In [ ]:
arranque = []
for h, d in historia.items():
    cohorte = d[(d.exceso_t == 0) & d.anio_objetivo.between(2018, 2024)]
    inicios = cohorte[cohorte.inicio == 1]
    arranque.append(
        {
            "horizonte": f"T+{h}",
            "inicios": len(inicios),
            "desde cero casos": int((inicios.casos_t == 0).sum()),
            "%": round(100 * (inicios.casos_t == 0).mean(), 1),
            "mediana de casos en el origen": float(inicios.casos_t.median()),
        }
    )
pd.DataFrame(arranque)

## 4. Estacionalidad

Los inicios no se reparten parejo en el año. El modelo recibe el mes del año
como predictor, así que parte de lo que aprende es esta forma.

In [ ]:
mes = panel.loc[:, ["municipality_code", "year_month", "dengue_notifications_count"]].copy()
mes["mes"] = mes.year_month.str[5:].astype(int)
por_mes = mes.groupby("mes").dengue_notifications_count.mean()

figura, eje = plt.subplots(figsize=(8, 3.5))
eje.bar(por_mes.index, por_mes.values)
eje.set_xticks(range(1, 13))
eje.set_xlabel("Mes del año")
eje.set_ylabel("Casos por municipio-mes (promedio)")
eje.set_title("Estacionalidad del dengue en Meta y Tolima, 2007-2024")
eje.grid(alpha=0.3, axis="y")
figura.tight_layout()
plt.show()

## 5. Qué tan desigual es la carga entre municipios

In [ ]:
por_municipio = (
    panel.groupby("municipality_code")
    .dengue_notifications_count.sum()
    .sort_values(ascending=False)
)
acumulado = por_municipio.cumsum() / por_municipio.sum()
print(f"{len(por_municipio)} municipios, {int(por_municipio.sum())} casos en total")
print(f"Los 10 municipios con más casos concentran el {100 * acumulado.iloc[9]:.1f} %")
print(f"La mitad de los casos está en los primeros {int((acumulado < 0.5).sum()) + 1} municipios")
por_municipio.head(10).to_frame("casos")

## Lectura

Tres cosas que condicionan todo lo que viene después:

1. La cohorte evaluable es bastante más chica que el panel, porque solo cuentan
   los municipios fuera de exceso en el mes de origen. Las métricas del informe
   se leen sobre esas filas.
2. Una parte importante de los inicios arranca sin casos notificados en el
   municipio. Ningún método basado en conteos puede ordenarlos, y eso fija un
   límite que no se mueve con un modelo mejor sino con información nueva.
3. La carga está muy concentrada en pocos municipios, lo que explica por qué el
   presupuesto de alertas se reparte globalmente entre los dos departamentos y
   no por cupos fijos.